# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Hello World

In [10]:
# First, let's print a simple message to ensure our environment is set up correctly.
print("Hello World")

Hello World


## 2. Initial Setup

In [11]:
import os
#os.chdir('..')
print("Current Working Directory " , os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability


In [7]:
#!pip show bitsandbytes
#!pip show acceleratec c -
#!pip show transformers
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

MemTotal: 1007.71 GB
MemFree: 25.63 GB
MemAvailable: 840.70 GB
Free GPU Memory (GB): 39.3936


In [ ]:
# Code formatting and linting

!black notebooks/Llama-3-8B-quant.ipynb
!pylint notebooks/Llama-3-8B-quant.ipynb

In [5]:
# HuggingFace authentication

import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the .env file
load_dotenv()

# Read the Hugging Face token from the environment variable
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

# Log in using the token
login(token=huggingface_token)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /nfs/homedirs/daro/.cache/huggingface/token
Login successful


In [12]:
# Checking CUDA availability

import torch

# Check CUDA availability
if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB


## 3. Loading Models

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"  # For instruction-based models.
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "microsoft/Phi-3-vision-128k-instruct"  # Small enough to run on a gpu_gtx1080.

model_family, model_identifier = model_name.split("/")

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)

memory_footprint = model.get_memory_footprint()
print(f"Model Memory Footprint: {(memory_footprint / (1024 ** 3)):.2f} GB")

Model Memory Footprint: 4.10 GB


In [ ]:
# Example inference

input_text = "What famous tower is in Paris?"
input_ids = tokenizer(input_text, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=input_ids["input_ids"],
    max_length=512,  # Adjust maximum output length
    num_beams=5,  # Adjust number of beams for beam search
)

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Generated Text:", generated_text)

In [ ]:
import json

results = {"input": input_text, "output": generated_text}
with open("results.json", "w") as f:
    json.dump(results, f)

## 4. Loading Datasets

In [14]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

directory_dataset = os.getcwd()
wikitext_batch_size = 1  # Just use batch size 1 for this project
#wikitext_batch_size = 16
#wikitext_batch_size = 64
sequence_length = 512  # Maximum sequence length - use the default value
wikitext_seed = 1

wikitext_data_module = WikiTextDataModule(
  directory_dataset=directory_dataset,
  batch_size=wikitext_batch_size,
  sequence_length=sequence_length,
  tokenizer_name=model_name,
  seed=wikitext_seed
)

#wikitext_train_dataloader = wikitext_data_module.train_dataloader()
wikitext_dataloader = wikitext_data_module.val_dataloader()

Token indices sequence length is longer than the specified maximum sequence length for this model (294896 > 2048). Running this sequence through the model will result in indexing errors


In [18]:
print("Length of datasets:", len(wikitext_data_module.train_dataset), len(wikitext_data_module.val_dataset), len(wikitext_data_module.test_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

dataset_size = len(wikitext_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


Length of datasets: 36718 3760 4358
Number of batches in train_dataloader: 575
Batch 1:
  Original Text:   = Homarus gammarus = 
   Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , 
  Input data (first 5 tokens): tensor([    1,   259,   353, 15089, 26465])
  Target labels (first 5 tokens): tensor([  259,   353, 15089, 26465, 24988])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])
Batch 2:
  Original Text: , with spots that coalesce , and yellow below . The red colour associated with lobsters only appears after cooking . This occurs b

In [15]:
wikitext_dataset = []

# Loop through each batch in the dataloader
for batch in wikitext_dataloader:
  # Assuming the batch contains input_ids (tokenized text) and labels
  input_ids, labels = batch
  
  # Decode the input IDs back to text using the tokenizer
  decoded_text = wikitext_data_module.tokenizer.decode(input_ids[0].tolist())  # Assuming first element in batch
  wikitext_dataset.append(decoded_text)
  
print(f"Sample texts from train dataloader: {wikitext_dataset[1][:1000]}")
print(f"Type: {type(wikitext_dataset)}, Length: {len(wikitext_dataset)}")


Sample texts from train dataloader: , with spots that coalesce , and yellow below . The red colour associated with lobsters only appears after cooking . This occurs because , in life , the red pigment astaxanthin is bound to a protein complex , but the complex is broken up by the heat of cooking , releasing the red pigment . 
  The closest relative of H. gammarus is the American lobster , Homarus americanus . The two species are very similar , and can be crossed artificially , although hybrids are unlikely to occur in the wild since their ranges do not overlap . The two species can be distinguished by a number of characteristics : 
  The rostrum of H. americanus bears one or more spines on the underside , which are lacking in H. gammarus . 
  The spines on the claws of H. americanus are red or red @-@ tipped , while those of H. gammarus are white or white @-@ tipped . 
  The underside of the claw of H. americanus is orange or red , while that of H. gammarus is creamy white or very pale

## 5. Quantization

### 5.1. BitsAndBytes

In [16]:
# BNB Config

import torch
from transformers import BitsAndBytesConfig

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# BnB Quantization Configurations
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)

bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

# Save paths
import os

bnb_8bit_model_path = os.path.join(MODEL_SAVE_PATH, "TinyLlama-1.1B-Chat-v1.0-bnb-8bit")  # Define your path
bnb_4bit_model_path = "TinyLlama-1.1B-Chat-v1.0-bnb-4bit"  # Define your path

In [9]:
!ls

LICENSE				   __init__.py	       pyproject.toml
README.md			   env-quant-rel.yaml  requirements.txt
TinyLlama-1.1B-Chat-v1.0-awq	   jupyter-5487.out    scripts
TinyLlama-1.1B-Chat-v1.0-bnb-4bit  notebooks	       setup.py
TinyLlama-1.1B-Chat-v1.0-bnb-8bit  outfiles	       src


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
# Quantization

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
# BnB Quantization - 8-bit
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype=torch.float32,
    device_map=device
)

print(f"8-bit Model Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

# BnB Quantization - 4-bit
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype=torch.float32,
    device_map=device
)

print(f"4-bit BNB Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")
from src.models.utils_llm import calculate_model_size
print(f"4-bit BNB Model Size: {calculate_model_size(awq_model_path)}")

8-bit Model Memory Footprint: 1.39 GB


KeyboardInterrupt: 

In [ ]:
# Save quantized models

bnb_8bit_model_path = "TinyLlama-1.1B-Chat-v1.0-bnb-8bit"  # Define your path
bnb_4bit_model_path = "TinyLlama-1.1B-Chat-v1.0-bnb-4bit"  # Define your path

model_bnb_8bit.save_pretrained(bnb_8bit_model_path)
tokenizer.save_pretrained(bnb_8bit_model_path)  # Saves tokenizer alongside

model_bnb_4bit.save_pretrained(bnb_4bit_model_path)
tokenizer.save_pretrained(bnb_4bit_model_path)  # Saves tokenizer alongside

print(f"8-bit BnB model saved at: {bnb_8bit_model_path}")
print(f"4-bit BnB model saved at: {bnb_4bit_model_path}")

### 5.2 AWQ

In [23]:
# AWQ Config
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# AWQ Calibration Split
awq_calib_split = "validation"
awq_model_path = f'{model_name.split("/")[1]}-awq'

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [24]:
# AWQ Quantization

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map=device
)

# Quantize with wikitext validation as calibration data
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=wikitext_dataset,  # Pass the loaded validation dataset here
)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
AWQ: 100%|██████████| 22/22 [03:21<00:00,  9.17s/it]


Model is quantized and saved at "TinyLlama-1.1B-Chat-v1.0-awq"
Total Model Size for TinyLlama-1.1B-Chat-v1.0-awq: 1.31 KB
Model size: None


In [ ]:
# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

from src.models.utils_llm import calculate_model_size
print(f"Model size: {calculate_model_size(awq_model_path)}")

In [ ]:
# Load model and generate text

awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Now you can use the loaded tokenizer and model for inference tasks
# For example, generating text:
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

### 5.3 HQQ

In [ ]:
# HQQ Config

from transformers import AutoTokenizer

# Define the model name and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 5.3.1 Option 1: All linear layers will use the same quantization config

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 1: All linear layers will use the same quantization config
quant_config_same = HqqConfig(
    nbits=8, 
    group_size=64, 
    quant_zero=False, 
    quant_scale=False, 
    axis=0  # Default value
)

# Quantize the model with the same config for all linear layers
model_same = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_same
)

# Save the quantized model with the same config for all layers
model_same_path = f"{model_name}-hqq-same"
model_same.save_pretrained(model_same_path)
tokenizer.save_pretrained(model_same_path)
print(f"Model with the same quantization config saved at '{model_same_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (same config): {calculate_model_size(model_same_path)}")

### 5.3.2. Option 2: Different configs for specific layers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig

# Option 2: Different configs for specific layers
q4_config = {'nbits': 4, 'group_size': 64, 'quant_zero': False, 'quant_scale': False}
q3_config = {'nbits': 3, 'group_size': 32, 'quant_zero': False, 'quant_scale': False}

quant_config_dynamic = HqqConfig(dynamic_config={
    'self_attn.q_proj': q4_config,
    'self_attn.k_proj': q4_config,
    'self_attn.v_proj': q4_config,
    'self_attn.o_proj': q4_config,
    'mlp.gate_proj': q3_config,
    'mlp.up_proj': q3_config,
    'mlp.down_proj': q3_config,
})

# Quantize the model with different configs for specific layers
model_dynamic = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_dynamic
)

# Save the quantized model with different configs for specific layers
model_dynamic_path = f"{model_name}-hqq-dynamic"
model_dynamic.save_pretrained(model_dynamic_path)
tokenizer.save_pretrained(model_dynamic_path)
print(f"Model with dynamic quantization config saved at '{model_dynamic_path}'")

from src.models.utils_llm import calculate_model_size
print(f"Model size (dynamic config): {calculate_model_size(model_dynamic_path)}")

## 6. Evaluation

### 6.1. Perplexity

In [ ]:
# Evaluate perplexity on each model
from src.evaluations.evaluate_text_generation import evaluate_perplexity

perplexity_8bit = evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)
perplexity_4bit = evaluate_perplexity(model_bnb_4bit, wikitext_data_module, device)
perplexity_original = evaluate_perplexity(model, wikitext_data_module, device)

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 6.2. Brier Score

In [2]:
import numpy as np
import evaluate

brier_score = evaluate.load("brier_score")
predictions = np.array([0, 0, 1, 1])
references = np.array([0.1, 0.9, 0.8, 0.3])
results = brier_score.compute(predictions=predictions, references=references)
print(results)

ValueError: Only binary classification is supported. The type of the target is continuous.

In [ ]:
def brier_score(Y, alpha):
    batch_size = alpha.size(0)

    p = torch.nn.functional.normalize(alpha, p=1, dim=-1)
    indices = torch.arange(batch_size)
    p[indices, Y.squeeze()] -= 1
    brier_score = p.norm(dim=-1).mean().cpu().detach().numpy()
    return brier_score
  